In [ ]:
import numpy as np
from sklearn.metrics import confusion_matrix 

'''
......(补充中间段代码)
'''
    def get_results(self):
        """Returns accuracy score evaluation result.
            - overall accuracy
            - mean accuracy
            - mean IU
            - fwavacc
        """
        hist = self.confusion_matrix
        acc = np.diag(hist).sum() / hist.sum() # np.diag(hist).sum() 计算混淆矩阵对角线（即正确分类的样本数）的总和。hist.sum() 计算混淆矩阵中所有元素的总和（即总样本数）
        acc_cls = np.diag(hist) / hist.sum(axis=1) # np.diag(hist) 提取混淆矩阵对角线的元素（每个类别的正确预测数）。hist.sum(axis=1) 计算混淆矩阵每一行的总和，表示每个真实类别的样本总数。
        cls_acc = dict(zip(range(self.n_classes), acc_cls))
        print('cls_acc', cls_acc)
        acc_cls = np.nanmean(acc_cls) 
        iu = np.diag(hist) / (hist.sum(axis=1) + hist.sum(axis=0) - np.diag(hist)) # iu 计算每个类别的交并比，即正确预测数除以真实正样本与预测正样本的总和。
        # 打印出来后的类似形式如下：这是在有 3 中类型的情况下
        # np.diag(hist)                                       [1.78732012e+08 6.01833100e+06 2.29575200e+06]
        # hist.sum(axis=1) + hist.sum(axis=0) - np.diag(hist) [2.1091579e+08 2.1040447e+07 1.9679268e+07]
        mean_iu = np.nanmean(iu) # mean_iu 计算所有类别交并比的均值，忽略 NaN 值。
        freq = hist.sum(axis=1) / hist.sum() # freq 计算每个类别的频率，即该类别样本数占总样本数的比例。
        fwavacc = (freq[freq > 0] * iu[freq > 0]).sum() # fwavacc 计算加权平均准确率，只有当类别频率大于0时才计算其对加权平均的贡献。
        cls_iu = dict(zip(range(self.n_classes), iu))
        print('cls_iu', cls_iu)

        # "每个类别交并比":type(iu),

        return {
                "Overall Acc": acc,
                "Mean Acc": acc_cls,
                "FreqW Acc": fwavacc,
                "Mean IoU": mean_iu,
                "Class IoU": cls_iu,
            }

In [ ]:
# 分割：

    def _fast_hist(self, label_true, label_pred):
        mask = (label_true >= 0) & (label_true < self.n_classes)

        # 输出形式示例：
        # label_true ===  [0 0 0 ... 2 2 0]
        # mask === [ True  True  True ...  True  True  True]

        # 当mask中的某个元素为True时，它表示label_true中对应位置的元素应该被选中；当为False时，则不应该被选中

        hist = np.bincount(
            self.n_classes * label_true[mask].astype(int) + label_pred[mask], # 将二维的（真实标签，预测标签）对映射到一维的索引上，其中每个索引都唯一地对应于一个（真实类别，预测类别）的组合。
            minlength=self.n_classes ** 2, # 输出数组的最小长度。有一个 n_classes x n_classes 的混淆矩阵（或直方图），所以最小长度应该是类别数的平方。
        ).reshape(self.n_classes, self.n_classes)

        # print('hist === ' , hist)
        # 输出形式如下：
        # hist ===  [[186694   2303    499]
        #             [ 16647  13212      0]
        #             [  9399      0   1646]]
        return hist

In [ ]:
    
        # ''',
        # self.meaniou = 0
        # self.ap = 0
        # self.ap_list = []
        # self.tps = []
        # self.fps = []
        # self.scores = []
        # '''.


# def _meaniou(self, label_true, label_pred):
    #     # (type(label_true)) # <class 'numpy.ndarray'>
    #     # (label_true.shape) # (230400,)

    #     # '''，
    #     iou_threshold = 0.5
    #     label_true = np.array(label_true)
    #     label_pred = np.array(label_pred)
    #     num_classes = self.n_classes
    #     height = 1
    #     width = len(label_true)
    #     # 在绘制mask时，设置阈值，
    #     """Calculate AP (Average Precision) for multi-class masks."""
    #     true_masks = self.create_masks_from_labels(num_classes, label_true, height, width)
    #     pred_masks = self.create_masks_from_labels(num_classes, label_pred, height, width)

    #     for cls in range(num_classes):
    #         for pred_mask in pred_masks:
    #             true_class_masks = [true_mask for true_mask in true_masks if np.any(true_mask == cls)]
    #             if not true_class_masks:
    #                 continue

    #             ious = [self.calculate_iou(true_mask, pred_mask) for true_mask in true_class_masks] # 
    #             max_iou = max(ious)
    #             if max_iou >= iou_threshold:
    #                 self.tps.append(1)
    #                 self.fps.append(0)
    #                 self.scores.append(max_iou)
    #             else:
    #                 self.tps.append(0)
    #                 self.fps.append(1)
    #                 self.scores.append(max_iou)

    #         # 根据上述得到一个矩阵。。。

    #         self.tps = np.array(self.tps)
    #         self.fps = np.array(self.fps)
    #         self.scores = np.array(self.scores)

    #         indices = np.argsort(self.scores)[::-1]
    #         self.tps = self.tps[indices]
    #         self.fps = self.fps[indices]

    #         cumulative_tps = np.cumsum(self.tps)
    #         cumulative_fps = np.cumsum(self.fps)

    #         precision = cumulative_tps / (cumulative_tps + cumulative_fps)
    #         recall = cumulative_tps / len(true_class_masks) if len(true_class_masks) > 0 else 0

    #         for i in range(len(precision) - 1):
    #             self.ap += (recall[i + 1] - recall[i]) * precision[i + 1]

    #         self.ap_list.append(self.ap)

    #     return np.mean(self.ap_list)

In [ ]:
        ap_list = []
        height = 1
        width = len(label_true)

    # for true_labels, pred_labels in zip(true_labels_list, pred_labels_list):
        true_masks = self.create_masks_from_labels(self.n_classes, label_true, height, width)
        pred_masks = self.create_masks_from_labels(self.n_classes, label_pred, height, width)

        for cls in range(self.n_classes):
            tps = []
            fps = []
            scores = []

            for pred_mask in pred_masks:
                true_class_masks = [true_mask for true_mask in true_masks if np.any(true_mask == cls)]
                if not true_class_masks:
                    continue

                ious = [self.calculate_iou(true_mask, pred_mask) for true_mask in true_class_masks]
                max_iou = max(ious)
                if max_iou >= 0.5:
                    # print('+++++++++++++++')
                    tps.append(1)
                    fps.append(0)
                    scores.append(max_iou)
                else:
                    # print('------------')
                    tps.append(0)
                    fps.append(1)
                    scores.append(max_iou)

            tps = np.array(tps)
            fps = np.array(fps)
            scores = np.array(scores)

            indices = np.argsort(scores)[::-1]
            tps = tps[indices]
            fps = fps[indices]

            cumulative_tps = np.cumsum(tps)
            cumulative_fps = np.cumsum(fps)

            precision = cumulative_tps / (cumulative_tps + cumulative_fps)
            recall = cumulative_tps / len(true_class_masks) if len(true_class_masks) > 0 else 0

            ap = 0.0
            for i in range(len(precision) - 1):
                ap += (recall[i + 1] - recall[i]) * precision[i + 1]

            ap_list.append(ap)